# Requirement 9 · LSTM and Transformer, latent space, attention, step / sigmoid / tanh

Sequence models on **real classroom video**: estimate the class's attentive share from the last 8 frames of the app's
own outputs (LSTM vs Transformer vs simple baselines), a Transformer that attends across the students of one frame,
the latent spaces of the ResNet and the LSTM, and the three LSTM cell non-linearities.
See `docs/guide/09_sequence_models.md`. **CPU runtime is enough** (about 5 minutes) once Requirement 7 has run.

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

In [ ]:
# needs the fine-tuned YOLOv8n and the ResNet18 state classifier from Requirement 7
import os
assert os.path.exists('models/yolo/yolov8n-kaggle.pt') and os.path.exists('models/classroom_states_resnet18.pt'),     'run notebook 07 first'

## 1 · Run everything

In [ ]:
!python -m smartattend sequence

In [ ]:
from IPython.display import Image, display
for name in ['classroom_lstm_transformer', 'classroom_attention', 'classroom_latent_space', 'lstm_activations']:
    display(Image(f'reports/figures/sequence/{name}.png'))

## 2 · An LSTM cell and a Transformer block, step by step

In [ ]:
import torch
from smartattend.sequence.lstm_trend import FlexLSTM
from smartattend.sequence.classroom_seq import TemporalTransformer, gradient_through_time

x = torch.randn(1, 8, 7)                       # 1 video window, 8 frames, 7 numbers per frame
for act in ['tanh', 'sigmoid', 'step']:
    h = FlexLSTM(7, 16, act)(x)                # hidden state after every frame
    g = gradient_through_time(act, T=20, trials=10)
    print(f'{act:8s} hidden states {tuple(h.shape)}  gradient reaching 19 frames back: {g[0] / g[-1]:.1e}')

y, attention = TemporalTransformer()(x, return_attn=True)
print('Transformer output', y.shape, ' attention over the 8 frames:', attention[0].detach().numpy().round(3), ' sum =', float(attention.sum()))

## 3 · Tests

In [ ]:
!python -m pytest -q tests/test_classroom_seq.py tests/test_vision_sequence.py